# 車外狀態機示範

```
車上 sender.send() → MQTT → receiver.py → cache.put("jetbot", payload) → 車外狀態機 → LINE（dry-run）
```

- 不用車子、不用 broker：以任務二的假資料情境（`transport/simulator.py`）當輸入，用假時鐘重播，幾分鐘的情境一瞬間跑完
- 分類表、冷卻、失聯秒數、訊息模板都在 `machine.py` 最上面的設定區
- 規則說明見 `docs/狀態機輸入格式.md`
- LINE 預設 dry-run，不會真的送出

在 repo 根目錄開啟：

```bash
uv run --with jupyter jupyter lab state_machine/demo.ipynb
```

In [ ]:
import os, sys
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "state_machine" else os.getcwd()
sys.path[:0] = [os.path.join(ROOT, "state_machine"), os.path.join(ROOT, "transport")]

import machine, simulator
from machine import StateMachine, format_message, line_output


class FakeClock(object):
    def __init__(self):
        self.t = 0.0
    def __call__(self):
        return self.t


def replay(scenario, tail_sec=1.0, show=True):
    """把一個情境的資料依時間寫進快取，車外每 0.1 秒判斷一拍。回傳狀態機。"""
    records = simulator.generate(scenario, fps=10, start_ts=0.0)
    clock = FakeClock()
    sm = StateMachine(outputs=[line_output(dry_run=True)], clock=clock, verbose=False)
    end_ms = int(round((simulator.scenario_duration(scenario) + tail_sec) * 1000))
    i = 0
    for now_ms in range(0, end_ms, 100):
        clock.t = now_ms / 1000.0
        while i < len(records) and int(round(records[i]["t"] * 1000)) <= now_ms:
            sm.cache.put("jetbot", records[i]["payload"])
            i += 1
        sm.tick()
    if show:
        print("收到 {} 筆，車上事件 {} 個，通知 {} 則".format(len(records), len(sm.log), len(sm.notices)))
        for n in sm.notices:
            print("\n── 第 {:.1f} 秒　{}／{} ──".format(n["time"], n["category"], n["kind"]))
            print(format_message(n))
    return sm

## 1. 卡住：`detour_stuck`

繞行 3 次都失敗 → 車上進入 `impassable_wait`，發 `enter_impassable`＋`notify_impassable`，之後每 60 秒重發。
車外只推一則（冷卻 5 分鐘），障礙移開、車上發 `resume_follow` 後推一則恢復。

In [ ]:
sm = replay("detour_stuck")

## 2. 看不到受護者：`long_loss`

跟丟 → 2 秒後 `lock_lost`（只記錄）→ 再 1 秒 `notify_no_target`（通知）。車上每 60 秒重發，車外不會每 60 秒推一則。

In [ ]:
sm = replay("long_loss")
print()
print("車上事件紀錄：")
for x in sm.log:
    print("  {:>6.1f}s  {:<18} state={}".format(x["ts"], x["event"], x["state"]))

## 3. 失聯：`offline`

車上照常運作，但資料送不出來。超過 5 秒沒收到 → 失聯；重新收到 → 恢復。

In [ ]:
sm = replay("offline")

## 4. 所有情境一覽

In [ ]:
for name in sorted(simulator.SCENARIOS):
    sm = replay(name, show=False)
    got = ", ".join("{:.1f}s {}/{}".format(n["time"], n["category"], n["kind"]) for n in sm.notices) or "不通知"
    print("{:<15} {}".format(name, got))

## 5. 給 LLM 的結構化事實（任務四）

訊息產生器是可替換的：任何 `fn(notice) -> str` 都能傳給 `line_output(formatter=...)`。
notice 本身就是結構化的事實，`notice_facts()` 去掉執行紀錄後可直接給 LLM。

In [ ]:
import json
sm = replay("detour_stuck", show=False)
print(json.dumps(machine.notice_facts(sm.notices[0]), ensure_ascii=False, indent=2))

my_line = line_output(dry_run=True, formatter=lambda n: "（自訂格式）{} {}".format(n["category"], n["kind"]))
print(my_line(sm.notices[0]))

## 6. 接真的 MQTT（需要 broker）

先在終端機啟動 broker（`cd transport/broker && docker compose up -d`），再執行下面這格；
另開終端機送假資料：`uv run --with "paho-mqtt<2.0" python transport/simulator.py detour_stuck --mqtt`。
本格需要 `paho-mqtt<2.0`：`uv run --with jupyter --with "paho-mqtt<2.0" jupyter lab state_machine/demo.ipynb`

In [ ]:
import receiver

live = StateMachine(outputs=[line_output(dry_run=True)], verbose=False)
rx = receiver.MqttReceiver(live.cache).start()
live.start()
# 之後隨時執行 print(live.status()) 查看

In [ ]:
print(live.status())

In [ ]:
live.stop(); rx.stop()